# Character-level RNN Text Generator
The aim of this project is to generate text based on the Shakespeare dataset.

The dataset represents a collection of text extracted from the works of William Shakespeare. It contains a variety of plays, sonnets, and his other literary works. The text is in English and includes a wide range of vocabulary and linguistic styles typical of Shakespearean literature.

### 1. Import Libraries

In [1]:
import requests

import torch
import torch.nn as nn
import torch.nn.functional as f
import torch.optim as optim

### 2. Load Dataset

In [2]:
# URL of the Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Fetching the text data from the URL
response = requests.get(url)
text = response.text

# Printing the length of the fetched text
print("Length of text: ", len(text))

Length of text:  1115394


### 3. Data Preparation

In [3]:
# Creating character mappings
chars = sorted(list(set(text)))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))

# Creating sequences
maxlen = 40
step = 5
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])
print("Number of sequences:", len(sentences))

Number of sequences: 223071


### 4. Data Vectorisation

In [4]:
# Vectorizing the data
X = torch.zeros((len(sentences), maxlen, len(chars)), dtype=torch.float32)
y = torch.zeros((len(sentences), len(chars)), dtype=torch.float32)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

# Check shape of vectors
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([223071, 40, 65])
y shape: torch.Size([223071, 65])


### 5. RNN Model

In [5]:
# Building the model
class RNNModel(nn.Module):
    """
    A class representing a simple recurrent neural network for
    character-level sequence prediction

    Attributes:
        input_size: number of unique characters in the vocabulary.
        hidden_size: number of neurons in the hidden layer.
        output_size: number of unique characters in the vocabulary.
        rnn: recurrent layer that processes the input sequence.
        fc: fully connected layer from hidden state to output layer.

    Methods:
        __init__(self, input_size, hidden_size, output_size)
            builds the network layers.
        forward(self, x)
            runs the forward pass through the network.
    """
    def __init__(self, input_size, hidden_size, output_size):
        super(RNNModel, self).__init__()
        # recurrent layer
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        # hidden to output
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # forward pass through the network
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        return out

In [6]:
# set random seed
torch.manual_seed(123)

# define input, hidden layer and output size
input_size = len(chars)
hidden_size = 128
output_size = len(chars)

# run model
model = RNNModel(input_size, hidden_size, output_size)

In [7]:
# define loss function and optimiser
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

In [8]:
# Training the model
num_epochs = 10
batch_size = 128
for epoch in range(num_epochs):
    for i in range(0, len(X), batch_size):
        X_batch = X[i:i+batch_size]
        y_batch = y[i:i+batch_size]

        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward pass and optimisation
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

Epoch [1/10], Loss: 2.2245
Epoch [2/10], Loss: 2.0654
Epoch [3/10], Loss: 1.9655
Epoch [4/10], Loss: 1.8855
Epoch [5/10], Loss: 1.8065
Epoch [6/10], Loss: 1.7444
Epoch [7/10], Loss: 1.6956
Epoch [8/10], Loss: 1.6577
Epoch [9/10], Loss: 1.6302
Epoch [10/10], Loss: 1.6093


Loss decreases steadily across all 10 epochs, from 2.2245 down to 1.6093, with no spikes or plateaus. This shows the model was learning consistently throughout training rather than getting stuck.

### 6. Text Generation

In [9]:
def vectorise_sequence(short_text, char_indices, maxlen, vocab_size):
    """
    Vectorise a text sequence into a one-hot encoded tensor.

    Parameters:
        short_text: string of characters to vectorise.
        char_indices: dictionary mapping characters to their index in the
                      vocabulary.
        maxlen: length of the character sequence window used as model input.
        vocab_size: number of unique characters in the vocabulary.

    Returns:
        A one-hot encoded tensor of shape (1, maxlen, vocab_size)
    """
    # empty tensor to hold one-hot encoded sequence
    encoded = torch.zeros((1, maxlen, vocab_size), dtype=torch.float32)

    # for each character in text, set corresponding index to 1
    for t, char in enumerate(short_text):
        encoded[0, t, char_indices[char]] = 1
    return encoded

In [10]:
def generate_text(model, seed_text, char_indices, indices_char,
                  maxlen, vocab_size, num_words=100):
    """
    Generates text from a trained character-level RNN model.

    Takes a seed text and repeatedly predicts the next character, sampling
    probabilistically until the specified number of words has been generated.

    Parameters:
        model: trained RNN model used to predict the next character.
        seed_text: starting string used as initial context, must be at least
                   maxlen characters long.
        char_indices: dictionary mapping characters to their index in the
                      vocabulary.
        indices_char: dictionary mapping integer indices back to their
                      corresponding characters.
        maxlen: length of the character sequence window used as model input.
        vocab_size: number of unique characters in the vocabulary.
        num_words: number of words to generate after the seed text.

    Returns:
        A string containing the seed text followed by the generated text.
    """
    # check length of string is valid
    if len(seed_text) < maxlen:
        raise ValueError(f'seed_text must be at least {maxlen}'
                         f' characters long')

    # initialise generated text with the seed text.
    generated_text = seed_text
    # count words in seed
    seed_w_count = len(seed_text.split())

    with torch.no_grad():
        while len(generated_text.split()) - seed_w_count < num_words:
            # get the current window of text and vectorise
            current_window = generated_text[-maxlen:]
            encoded_text = vectorise_sequence(current_window, char_indices,
                                              maxlen, vocab_size)

            # forward pass
            output = model(encoded_text)

            # apply softmax to get probabilities
            probabilities = f.softmax(output, dim=1)

            # select next char based on probabilities
            next_index = torch.multinomial(probabilities, 1).item()
            next_char = indices_char[next_index]

            # add next char to generated text
            generated_text += next_char

        return generated_text

In [11]:
# use first 40 characters of the text as the seed
seed_text = text[:maxlen]

# generate 100 words of text using function
result = generate_text(model, seed_text, char_indices, indices_char,
                       maxlen, len(chars))
print(result)

First Citizen:
Before we proceed any fure uneng o this marrabest of you llad the of myshed.

TIONIE:
I wiscall the rensty
For ain to
Two, wistans.

COPURENCASTORD:
My lans!

CUSEY:

PENTANRY:
I smoun painted
os pomman
Go sting illoxs, and appeate to theme-poom as I is oll oy wind speak sut 'twill, farricg'd
cition the ershcals it genty stoy:
'Tis if this suck and and ampell.

ANTONTCANIO:
O can would commons! felse is calf and ullorn,
I wis trued your sevence.

GROMIO:
Not yave buraght.

FRIAD: I'll ring swed, he stunt of not think to be:
To fere, wince, my lare.

MIOSPELANG ONPETHA:
O to-Nour h


The generated text follows the script structure well with character names, colons, and line breaks, matching the original format. The actual wording doesn't fully make sense though, real words show up here and there, but they're mixed with invented ones, and sentences don't hold together grammatically. The model would need to be refined a lot further to improve upon this.